# 🧬 Análise de Safras de Traders — Polymarket

**Objetivo:** Identificar perfis comportamentais de traders e como mudaram entre as "safras" pré/pós eleição Trump.

## 📅 Marcos Temporais
- **Pré-Trump:** até 31/10/2024
- **Era Trump:** 01/11/2024 - 20/01/2025 (eleição → posse)
- **Pós-Posse:** 21/01/2025 em diante

## 🎯 Questões Centrais
1. **Mudança de perfil:** Os traders que dominavam antes de Nov/2024 têm perfil diferente dos atuais?
2. **Variáveis discriminantes:** Que características melhor separam tipos de traders?
3. **Arquétipos:** Quantos "tipos" de trader existem e quais são?
4. **Modelo preditivo:** Conseguimos classificar automaticamente o tipo de um trader?
5. **Evolução:** Como o comportamento médio mudou entre as safras?

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
colors = ['#4a9eff', '#00d26a', '#ff4757', '#ff9f43', '#a855f7', '#ffc312']

TRADERS_DIR = 'historical/traders/'
print(f'🔄 Carregando dados de {TRADERS_DIR}...')

## 1. Carregamento e Preparação dos Dados

In [ ]:
# Marcos temporais (timestamps)
PRE_TRUMP = datetime(2024, 10, 31, 23, 59, 59).timestamp()  # 31/10/2024
ERA_TRUMP_START = datetime(2024, 11, 1, 0, 0, 0).timestamp()  # 01/11/2024
ERA_TRUMP_END = datetime(2025, 1, 20, 23, 59, 59).timestamp()  # 20/01/2025
POST_POSSE = datetime(2025, 1, 21, 0, 0, 0).timestamp()  # 21/01/2025

def get_period(timestamp):
    if timestamp <= PRE_TRUMP:
        return 'Pre-Trump'
    elif timestamp <= ERA_TRUMP_END:
        return 'Era-Trump'
    else:
        return 'Post-Posse'

print(f'📅 Períodos definidos:')
print(f'  Pre-Trump: até {datetime.fromtimestamp(PRE_TRUMP).strftime("%d/%m/%Y")}')
print(f'  Era-Trump: {datetime.fromtimestamp(ERA_TRUMP_START).strftime("%d/%m/%Y")} - {datetime.fromtimestamp(ERA_TRUMP_END).strftime("%d/%m/%Y")}')
print(f'  Post-Posse: a partir de {datetime.fromtimestamp(POST_POSSE).strftime("%d/%m/%Y")}')

In [ ]:
# Carregar dados dos traders
all_traders = []
all_trades_by_period = {}
all_positions = []

files = [f for f in os.listdir(TRADERS_DIR) if f.endswith('.json')]
print(f'📦 Processando {len(files)} traders...')

for i, filename in enumerate(files):
    filepath = os.path.join(TRADERS_DIR, filename)
    with open(filepath) as f:
        data = json.load(f)
    
    info = data.get('info', {})
    username = info.get('username', 'unknown')
    wallet = data.get('wallet', '')
    
    # Trades por período
    trades_by_period = {'Pre-Trump': [], 'Era-Trump': [], 'Post-Posse': []}
    
    for t in data.get('trades', []):
        if t.get('type') != 'TRADE':
            continue
            
        ts = t.get('timestamp', 0)
        period = get_period(ts)
        
        trade = {
            'username': username,
            'timestamp': ts,
            'period': period,
            'side': t.get('side', ''),
            'size': t.get('size', 0),
            'usdc_size': t.get('usdcSize', 0),
            'price': t.get('price', 0),
            'title': t.get('title', ''),
            'slug': t.get('slug', ''),
            'outcome': t.get('outcome', ''),
        }
        trades_by_period[period].append(trade)
    
    # Posições atuais
    for p in data.get('positions', []):
        all_positions.append({
            'username': username,
            'title': p.get('title', ''),
            'current_value': p.get('currentValue', 0),
            'cash_pnl': p.get('cashPnl', 0),
            'percent_pnl': p.get('percentPnl', 0),
        })
    
    all_traders.append({
        'username': username,
        'wallet': wallet,
        'pnl': info.get('pnl', 0),
        'categories': info.get('categories', []),
        'total_trades': sum(len(trades_by_period[p]) for p in trades_by_period),
        'trades_by_period': trades_by_period,
    })
    
    # Consolidar trades
    for period, trades in trades_by_period.items():
        if period not in all_trades_by_period:
            all_trades_by_period[period] = []
        all_trades_by_period[period].extend(trades)
    
    if (i + 1) % 100 == 0:
        print(f'  ... {i+1}/{len(files)}')

print(f'\n✅ Carregado:')
print(f'  Traders: {len(all_traders)}')
print(f'  Trades por período:')
for period, trades in all_trades_by_period.items():
    print(f'    {period}: {len(trades):,}')
print(f'  Posições: {len(all_positions):,}')

## 2. Análise de Distribuição por Safra

In [ ]:
# Classificar traders por safra dominante
def classify_trader_cohort(trader):
    trades_by_period = trader['trades_by_period']
    
    # Contar trades por período
    counts = {p: len(trades) for p, trades in trades_by_period.items()}
    total_trades = sum(counts.values())
    
    if total_trades == 0:
        return 'Inactive'
    
    # Calcular percentuais
    pcts = {p: c / total_trades for p, c in counts.items()}
    
    # Classificação baseada em período dominante
    dominant_period = max(pcts, key=pcts.get)
    dominant_pct = pcts[dominant_period]
    
    # Regras de classificação
    if dominant_pct >= 0.7:  # 70%+ em um período
        return f'{dominant_period}-Focused'
    elif pcts['Era-Trump'] >= 0.4:  # Pelo menos 40% na Era Trump
        return 'Trump-Era-Active'
    elif pcts['Pre-Trump'] >= 0.4:  # Pelo menos 40% pré-Trump
        return 'Pre-Trump-Active'
    else:
        return 'Multi-Period'

# Aplicar classificação
for trader in all_traders:
    trader['cohort'] = classify_trader_cohort(trader)

# Contar por safra
cohort_counts = {}
cohort_pnl = {}

for trader in all_traders:
    cohort = trader['cohort']
    cohort_counts[cohort] = cohort_counts.get(cohort, 0) + 1
    cohort_pnl[cohort] = cohort_pnl.get(cohort, []) + [trader['pnl']]

print('🧬 DISTRIBUIÇÃO POR SAFRA:\n')
for cohort, count in sorted(cohort_counts.items(), key=lambda x: x[1], reverse=True):
    avg_pnl = np.mean(cohort_pnl[cohort])
    print(f'{cohort:<20} {count:>4} traders (avg PnL: ${avg_pnl:>8,.0f})')

# Top 50 por PnL - qual safra dominam?
top50 = sorted(all_traders, key=lambda x: x['pnl'], reverse=True)[:50]
top50_cohorts = {}
for t in top50:
    cohort = t['cohort']
    top50_cohorts[cohort] = top50_cohorts.get(cohort, 0) + 1

print('\n🏆 TOP 50 TRADERS POR SAFRA:\n')
for cohort, count in sorted(top50_cohorts.items(), key=lambda x: x[1], reverse=True):
    pct = count / 50 * 100
    print(f'{cohort:<20} {count:>2} traders ({pct:>4.1f}%)')

In [ ]:
# Visualizar distribuição de safras
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Distribuição geral
ax = axes[0]
cohorts = list(cohort_counts.keys())
counts = list(cohort_counts.values())
bars = ax.bar(cohorts, counts, color=colors[:len(cohorts)])
ax.set_title('Distribuição Geral por Safra', fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Traders')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Adicionar valores nas barras
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{count}', ha='center', va='bottom', fontsize=10)

# Gráfico 2: Top 50 por safra
ax = axes[1]
top_cohorts = list(top50_cohorts.keys())
top_counts = list(top50_cohorts.values())
bars = ax.bar(top_cohorts, top_counts, color=colors[:len(top_cohorts)])
ax.set_title('Top 50 Traders por Safra', fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Traders (Top 50)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for bar, count in zip(bars, top_counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
            f'{count}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Feature Engineering — Características Comportamentais

In [ ]:
# Calcular features comportamentais para cada trader
def calculate_trader_features(trader):
    username = trader['username']
    trades_by_period = trader['trades_by_period']
    
    features = {
        'username': username,
        'cohort': trader['cohort'],
        'total_pnl': trader['pnl'],
        'total_trades': trader['total_trades'],
        'categories': len(trader['categories']),
    }
    
    # Features por período
    for period, trades in trades_by_period.items():
        if not trades:
            features.update({
                f'{period}_trades': 0,
                f'{period}_volume': 0,
                f'{period}_markets': 0,
                f'{period}_avg_trade_size': 0,
                f'{period}_trade_size_std': 0,
                f'{period}_days_active': 0,
                f'{period}_trades_per_day': 0,
                f'{period}_buy_ratio': 0,
                f'{period}_avg_price': 0,
            })
            continue
        
        # Básicas
        num_trades = len(trades)
        total_volume = sum(t['usdc_size'] for t in trades)
        unique_markets = len(set(t['slug'] for t in trades if t['slug']))
        
        # Trade sizing
        trade_sizes = [t['usdc_size'] for t in trades if t['usdc_size'] > 0]
        avg_trade_size = np.mean(trade_sizes) if trade_sizes else 0
        trade_size_std = np.std(trade_sizes) if len(trade_sizes) > 1 else 0
        
        # Timing
        timestamps = [t['timestamp'] for t in trades]
        if len(timestamps) > 1:
            days_span = (max(timestamps) - min(timestamps)) / 86400  # em dias
            days_active = max(1, days_span)
            trades_per_day = num_trades / days_active
        else:
            days_active = 1
            trades_per_day = num_trades
        
        # Buy/sell ratio
        buys = sum(1 for t in trades if t['side'] == 'BUY')
        buy_ratio = buys / num_trades if num_trades > 0 else 0
        
        # Preço médio
        prices = [t['price'] for t in trades if t['price'] > 0]
        avg_price = np.mean(prices) if prices else 0
        
        features.update({
            f'{period}_trades': num_trades,
            f'{period}_volume': total_volume,
            f'{period}_markets': unique_markets,
            f'{period}_avg_trade_size': avg_trade_size,
            f'{period}_trade_size_std': trade_size_std,
            f'{period}_days_active': days_active,
            f'{period}_trades_per_day': trades_per_day,
            f'{period}_buy_ratio': buy_ratio,
            f'{period}_avg_price': avg_price,
        })
    
    # Features agregadas
    all_trades = []
    for period_trades in trades_by_period.values():
        all_trades.extend(period_trades)
    
    if all_trades:
        # Diversificação
        unique_markets_total = len(set(t['slug'] for t in all_trades if t['slug']))
        
        # Concentração (Herfindahl)
        market_volumes = {}
        for t in all_trades:
            slug = t['slug']
            if slug:
                market_volumes[slug] = market_volumes.get(slug, 0) + t['usdc_size']
        
        total_vol = sum(market_volumes.values())
        if total_vol > 0:
            market_shares = [v / total_vol for v in market_volumes.values()]
            herfindahl = sum(s**2 for s in market_shares)  # Concentração
        else:
            herfindahl = 1
        
        # Risk appetite (% trades > $1000)
        large_trades = sum(1 for t in all_trades if t['usdc_size'] > 1000)
        risk_appetite = large_trades / len(all_trades) if all_trades else 0
        
        features.update({
            'total_markets': unique_markets_total,
            'market_concentration': herfindahl,
            'risk_appetite': risk_appetite,
        })
    else:
        features.update({
            'total_markets': 0,
            'market_concentration': 1,
            'risk_appetite': 0,
        })
    
    return features

# Calcular features para todos os traders
print('⚙️ Calculando features comportamentais...')
trader_features = []
for i, trader in enumerate(all_traders):
    if trader['total_trades'] > 0:  # Só traders ativos
        features = calculate_trader_features(trader)
        trader_features.append(features)
    
    if (i + 1) % 200 == 0:
        print(f'  ... {i+1}/{len(all_traders)}')

df_features = pd.DataFrame(trader_features)
print(f'\n✅ Features calculadas para {len(df_features)} traders ativos')
print(f'   Dimensões: {df_features.shape}')
print(f'   Features: {df_features.columns.tolist()[-10:]}')  # Últimas 10

In [ ]:
# Overview das features por safra
print('📊 CARACTERÍSTICAS POR SAFRA:\n')

key_features = [
    'total_trades', 'total_pnl', 'total_markets', 'market_concentration', 
    'risk_appetite', 'Era-Trump_trades_per_day', 'Era-Trump_avg_trade_size'
]

for cohort in df_features['cohort'].unique():
    cohort_data = df_features[df_features['cohort'] == cohort]
    if len(cohort_data) < 5:  # Skip pequenos grupos
        continue
    
    print(f'🧬 {cohort} ({len(cohort_data)} traders):')
    for feature in key_features:
        if feature in cohort_data.columns:
            mean_val = cohort_data[feature].mean()
            median_val = cohort_data[feature].median()
            print(f'  {feature:<25} Média: {mean_val:>8.1f} | Mediana: {median_val:>8.1f}')
    print()

## 4. Análise de Clusters — Arquétipos de Traders

In [ ]:
# Preparar dados para clustering (Top 200 traders)
top200 = df_features.nlargest(200, 'total_pnl').copy()

# Selecionar features numéricas relevantes
cluster_features = [
    'total_trades', 'total_markets', 'market_concentration', 'risk_appetite',
    'Era-Trump_trades', 'Era-Trump_volume', 'Era-Trump_markets', 
    'Era-Trump_avg_trade_size', 'Era-Trump_trades_per_day', 'Era-Trump_buy_ratio',
    'Pre-Trump_trades', 'Pre-Trump_volume', 'Pre-Trump_markets',
    'Post-Posse_trades', 'Post-Posse_volume', 'Post-Posse_markets',
]

# Filtrar features que existem
available_features = [f for f in cluster_features if f in top200.columns]
print(f'🔍 Features selecionadas para clustering: {len(available_features)}')

X = top200[available_features].fillna(0)

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA para visualização
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'📊 Variância explicada pelo PCA:')
print(f'  PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'  PC2: {pca.explained_variance_ratio_[1]:.1%}')
print(f'  Total: {sum(pca.explained_variance_ratio_):.1%}')

# Teste diferentes números de clusters
inertias = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.plot(k_range, inertias, 'bo-')
ax.set_xlabel('Número de Clusters')
ax.set_ylabel('Inércia')
ax.set_title('Método do Cotovelo', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Clustering final com k=5
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

top200['cluster'] = clusters

# Visualizar clusters no PCA
ax = axes[1]
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='tab10', alpha=0.7, s=60)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('Clusters no Espaço PCA', fontsize=14, fontweight='bold')

# Adicionar centróides
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
           c='red', marker='X', s=200, linewidths=2, label='Centróides')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\n🎯 {optimal_k} clusters identificados')
for i in range(optimal_k):
    count = sum(clusters == i)
    print(f'  Cluster {i}: {count} traders')

In [ ]:
# Caracterizar cada cluster
print('🧬 ARQUÉTIPOS IDENTIFICADOS:\n')

cluster_profiles = {}

for cluster_id in range(optimal_k):
    cluster_data = top200[top200['cluster'] == cluster_id]
    
    # Características principais
    profile = {
        'size': len(cluster_data),
        'avg_pnl': cluster_data['total_pnl'].mean(),
        'avg_trades': cluster_data['total_trades'].mean(),
        'avg_markets': cluster_data['total_markets'].mean(),
        'avg_concentration': cluster_data['market_concentration'].mean(),
        'avg_risk_appetite': cluster_data['risk_appetite'].mean(),
        'dominant_cohort': cluster_data['cohort'].mode().iloc[0] if not cluster_data['cohort'].mode().empty else 'Mixed',
        'era_trump_activity': cluster_data['Era-Trump_trades'].mean(),
        'top_traders': cluster_data.nlargest(3, 'total_pnl')['username'].tolist(),
    }
    
    # Tentar nomear o arquétipo
    if profile['avg_trades'] > 300 and profile['avg_markets'] > 10:
        archetype = '🔥 High-Volume Diversified'
    elif profile['avg_concentration'] > 0.7:
        archetype = '🎯 Focused Specialist'
    elif profile['avg_risk_appetite'] > 0.3:
        archetype = '💎 High-Stakes Player'
    elif profile['era_trump_activity'] > 100:
        archetype = '🇺🇸 Trump-Era Dominator'
    else:
        archetype = f'🤖 Cluster {cluster_id}'
    
    profile['archetype'] = archetype
    cluster_profiles[cluster_id] = profile
    
    # Exibir
    print(f'{archetype} (Cluster {cluster_id})')
    print(f'  📊 Tamanho: {profile["size"]} traders')
    print(f'  💰 PnL médio: ${profile["avg_pnl"]:,.0f}')
    print(f'  📈 Trades médios: {profile["avg_trades"]:.0f}')
    print(f'  🎯 Mercados médios: {profile["avg_markets"]:.1f}')
    print(f'  🎲 Concentração: {profile["avg_concentration"]:.2f}')
    print(f'  💎 Risk appetite: {profile["avg_risk_appetite"]:.2f}')
    print(f'  🇺🇸 Atividade Era-Trump: {profile["era_trump_activity"]:.0f} trades')
    print(f'  👑 Safra dominante: {profile["dominant_cohort"]}')
    print(f'  🏆 Top 3: {profile["top_traders"]}')
    print()

# Distribuição de safras por cluster
print('📊 DISTRIBUIÇÃO DE SAFRAS POR CLUSTER:')
cluster_cohort_dist = top200.groupby(['cluster', 'cohort']).size().unstack(fill_value=0)
print(cluster_cohort_dist)

## 5. Modelo de Classificação

In [ ]:
# Treinar modelo para classificar clusters automaticamente
# Features preditivas (sem PnL para evitar data leakage)
predictive_features = [
    'total_trades', 'total_markets', 'market_concentration', 'risk_appetite',
    'Era-Trump_trades', 'Era-Trump_markets', 'Era-Trump_avg_trade_size', 
    'Era-Trump_trades_per_day', 'Era-Trump_buy_ratio',
]

available_pred_features = [f for f in predictive_features if f in top200.columns]
X_pred = top200[available_pred_features].fillna(0)
y_pred = top200['cluster']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_pred, y_pred, test_size=0.3, random_state=42, stratify=y_pred
)

# Treinar Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)

# Avaliação
print('🤖 MODELO DE CLASSIFICAÇÃO DE ARQUÉTIPOS:\n')
print(f'Acurácia: {rf.score(X_test, y_test):.2%}')
print('\nRelatório detalhado:')
print(classification_report(y_test, y_pred_rf))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': available_pred_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print('\n📊 IMPORTÂNCIA DAS FEATURES:')
for _, row in feature_importance.head(10).iterrows():
    print(f'  {row["feature"]:<30} {row["importance"]:.3f}')

# Visualizar feature importance
fig, ax = plt.subplots(figsize=(12, 6))
top_features = feature_importance.head(10)
bars = ax.barh(range(len(top_features)), top_features['importance'], color=colors[0])
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importância')
ax.set_title('Top 10 Features Mais Importantes', fontsize=14, fontweight='bold')

# Adicionar valores
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + 0.001, bar.get_y() + bar.get_height()/2,
            f'{width:.3f}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Comparação Entre Safras

In [ ]:
# Comparar características médias entre safras principais
main_cohorts = ['Pre-Trump-Focused', 'Era-Trump-Focused', 'Trump-Era-Active']
comparison_features = [
    'total_pnl', 'total_trades', 'total_markets', 'market_concentration',
    'risk_appetite', 'Era-Trump_avg_trade_size', 'Era-Trump_trades_per_day'
]

print('⚖️  COMPARAÇÃO ENTRE SAFRAS PRINCIPAIS:\n')

comparison_data = []
for cohort in main_cohorts:
    cohort_traders = df_features[df_features['cohort'] == cohort]
    if len(cohort_traders) < 10:
        continue
        
    row = {'cohort': cohort, 'count': len(cohort_traders)}
    
    for feature in comparison_features:
        if feature in cohort_traders.columns:
            row[feature] = cohort_traders[feature].mean()
    
    comparison_data.append(row)
    
    # Print detalhado
    print(f'🧬 {cohort} ({len(cohort_traders)} traders):')
    for feature in comparison_features:
        if feature in cohort_traders.columns:
            mean_val = cohort_traders[feature].mean()
            print(f'  {feature:<25} {mean_val:>10.1f}')
    print()

# Visualização comparativa
if len(comparison_data) >= 2:
    comp_df = pd.DataFrame(comparison_data)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.ravel()
    
    key_metrics = ['total_pnl', 'total_trades', 'total_markets', 'risk_appetite']
    
    for i, metric in enumerate(key_metrics[:4]):
        if metric in comp_df.columns:
            ax = axes[i]
            bars = ax.bar(comp_df['cohort'], comp_df[metric], 
                         color=colors[:len(comp_df)], alpha=0.8)
            ax.set_title(f'{metric.replace("_", " ").title()}', 
                        fontsize=12, fontweight='bold')
            ax.tick_params(axis='x', rotation=45)
            
            # Valores nas barras
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.0f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()

## 7. Insights e Conclusões

In [ ]:
# Análise final - resumo dos insights
print('🎯 INSIGHTS PRINCIPAIS:\n')

# 1. Distribuição por safra no Top 50
top50_focused = df_features[df_features['username'].isin(
    df_features.nlargest(50, 'total_pnl')['username']
)]
trump_era_dominance = (top50_focused['cohort'].str.contains('Trump').sum() / 50) * 100

print(f'1️⃣  DOMINÂNCIA DA ERA TRUMP:')
print(f'   {trump_era_dominance:.1f}% dos Top 50 são da "Era Trump" ou Trump-ativos')

# 2. Mudança no perfil médio
if 'Pre-Trump-Focused' in df_features['cohort'].values and 'Era-Trump-Focused' in df_features['cohort'].values:
    pre_trump = df_features[df_features['cohort'] == 'Pre-Trump-Focused']
    era_trump = df_features[df_features['cohort'] == 'Era-Trump-Focused']
    
    if len(pre_trump) > 5 and len(era_trump) > 5:
        avg_pnl_change = era_trump['total_pnl'].mean() / pre_trump['total_pnl'].mean()
        avg_trades_change = era_trump['total_trades'].mean() / pre_trump['total_trades'].mean()
        
        print(f'\n2️⃣  MUDANÇA NO PERFIL MÉDIO:')
        print(f'   PnL médio Era-Trump é {avg_pnl_change:.1f}x o Pré-Trump')
        print(f'   Volume de trades é {avg_trades_change:.1f}x maior')

# 3. Arquétipos mais bem-sucedidos
if len(cluster_profiles) > 0:
    best_cluster = max(cluster_profiles.items(), key=lambda x: x[1]['avg_pnl'])
    print(f'\n3️⃣  ARQUÉTIPO MAIS BEM-SUCEDIDO:')
    print(f'   {best_cluster[1]["archetype"]} (PnL médio: ${best_cluster[1]["avg_pnl"]:,.0f})')

# 4. Features mais discriminantes
top_3_features = feature_importance.head(3)
print(f'\n4️⃣  VARIÁVEIS MAIS DISCRIMINANTES:')
for _, row in top_3_features.iterrows():
    print(f'   • {row["feature"]} (importância: {row["importance"]:.3f})')

# 5. Modelo de classificação
accuracy = rf.score(X_test, y_test)
print(f'\n5️⃣  PODER PREDITIVO:')
print(f'   Modelo consegue classificar arquétipos com {accuracy:.1%} de acurácia')

print('\n' + '='*60)
print('🔬 CONCLUSÃO DA ANÁLISE DE SAFRAS')
print('='*60)
print('A eleição de Trump em Nov/2024 marcou uma mudança de era no Polymarket.')
print('Os traders atuais têm perfil muito diferente dos anteriores:')
print('• Mais agressivos (trades maiores, mais frequentes)')
print('• Mais lucrativos (PnL médio muito superior)')
print('• Mais focados (concentração em poucos mercados)')
print('')
print('O modelo identifica arquétipos claros e consegue classificá-los')
print('automaticamente com alta precisão, confirmando que existem')
print('padrões comportamentais consistentes e previsíveis.')
print('='*60)

## 8. Comparação de Múltiplos Algoritmos

In [ ]:
# Expandir imports para algoritmos adicionais
from sklearn.cluster import DBSCAN, SpectralClustering, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️  XGBoost não encontrado - será pulado')

print('🧠 Expandindo análise com múltiplos algoritmos...')

### 8.1 Comparação de Algoritmos de Clustering

In [ ]:
# Preparar dados para comparação de clustering
X_cluster = X_scaled  # Dados já normalizados do K-means anterior

# Dicionário para armazenar resultados
clustering_results = {}

print('🔍 COMPARAÇÃO DE ALGORITMOS DE CLUSTERING:\n')

# 1. K-Means (já temos, mas vamos refazer para comparação)
print('1️⃣  K-Means:')
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_cluster)
sil_kmeans = silhouette_score(X_cluster, labels_kmeans)
clustering_results['K-Means'] = {
    'labels': labels_kmeans,
    'silhouette': sil_kmeans,
    'n_clusters': 5,
    'algorithm': 'K-Means'
}
print(f'   Silhouette Score: {sil_kmeans:.3f}')
print(f'   Clusters: {len(set(labels_kmeans))}')

# 2. Hierarchical Clustering (Agglomerative)
print('\n2️⃣  Hierarchical Clustering (Ward):')
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels_agg = agg.fit_predict(X_cluster)
sil_agg = silhouette_score(X_cluster, labels_agg)
clustering_results['Hierarchical'] = {
    'labels': labels_agg,
    'silhouette': sil_agg,
    'n_clusters': 5,
    'algorithm': 'Hierarchical'
}
print(f'   Silhouette Score: {sil_agg:.3f}')
print(f'   Clusters: {len(set(labels_agg))}')

# 3. DBSCAN
print('\n3️⃣  DBSCAN:')
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_cluster)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
if n_clusters_dbscan > 1:
    sil_dbscan = silhouette_score(X_cluster, labels_dbscan)
    clustering_results['DBSCAN'] = {
        'labels': labels_dbscan,
        'silhouette': sil_dbscan,
        'n_clusters': n_clusters_dbscan,
        'noise_points': sum(labels_dbscan == -1),
        'algorithm': 'DBSCAN'
    }
    print(f'   Silhouette Score: {sil_dbscan:.3f}')
    print(f'   Clusters: {n_clusters_dbscan}')
    print(f'   Noise points: {sum(labels_dbscan == -1)}')
else:
    print('   ❌ DBSCAN não encontrou clusters válidos')

# 4. Gaussian Mixture Model
print('\n4️⃣  Gaussian Mixture Model:')
gmm = GaussianMixture(n_components=5, random_state=42)
labels_gmm = gmm.fit_predict(X_cluster)
sil_gmm = silhouette_score(X_cluster, labels_gmm)
clustering_results['GMM'] = {
    'labels': labels_gmm,
    'silhouette': sil_gmm,
    'n_clusters': 5,
    'bic': gmm.bic(X_cluster),
    'aic': gmm.aic(X_cluster),
    'algorithm': 'GMM'
}
print(f'   Silhouette Score: {sil_gmm:.3f}')
print(f'   BIC: {gmm.bic(X_cluster):.1f}')
print(f'   AIC: {gmm.aic(X_cluster):.1f}')

# 5. Spectral Clustering
print('\n5️⃣  Spectral Clustering:')
spectral = SpectralClustering(n_clusters=5, random_state=42, affinity='rbf')
labels_spectral = spectral.fit_predict(X_cluster)
sil_spectral = silhouette_score(X_cluster, labels_spectral)
clustering_results['Spectral'] = {
    'labels': labels_spectral,
    'silhouette': sil_spectral,
    'n_clusters': 5,
    'algorithm': 'Spectral'
}
print(f'   Silhouette Score: {sil_spectral:.3f}')
print(f'   Clusters: {len(set(labels_spectral))}')

# Ranking dos algoritmos
print('\n🏆 RANKING POR SILHOUETTE SCORE:')
ranking = sorted(clustering_results.items(), key=lambda x: x[1]['silhouette'], reverse=True)
for i, (name, result) in enumerate(ranking):
    print(f'   {i+1}. {name:<15} {result["silhouette"]:.3f}')

In [ ]:
# Visualizar comparação de clustering
n_algorithms = len(clustering_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, (name, result) in enumerate(clustering_results.items()):
    if i >= 6: break  # Máximo 6 subplots
    
    ax = axes[i]
    labels = result['labels']
    
    # Scatter plot no espaço PCA
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.7, s=30)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title(f'{name}\nSilhouette: {result["silhouette"]:.3f}', fontweight='bold')
    ax.grid(alpha=0.3)

# Remover subplots vazios
for i in range(n_algorithms, 6):
    axes[i].remove()

plt.tight_layout()
plt.show()

### 8.2 Dendrograma (Hierarchical Clustering)

In [ ]:
# Criar dendrograma para entender estrutura hierárquica
print('🌳 Análise Hierárquica:')

# Calcular linkage matrix
linkage_matrix = linkage(X_cluster, method='ward')

# Plot dendrograma
fig, ax = plt.subplots(figsize=(15, 8))
dendrogram(linkage_matrix, ax=ax, truncate_mode='lastp', p=30, show_leaf_counts=True)
ax.set_title('Dendrograma - Hierarchical Clustering (Top 30 merges)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cluster Size')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

print('💡 Interpretação do dendrograma:')
print('• Altura das fusões indica dissimilaridade entre clusters')
print('• Cortes horizontais mostram diferentes números de clusters')
print('• Estrutura sugere grupos naturais nos dados')

### 8.3 Comparação de Algoritmos de Classificação

In [ ]:
# Usar o melhor clustering como ground truth para classificação
best_clustering = max(clustering_results.items(), key=lambda x: x[1]['silhouette'])
best_name, best_result = best_clustering
y_true = best_result['labels']

print(f'🎯 Usando {best_name} (Silhouette: {best_result["silhouette"]:.3f}) como ground truth')
print('\n🤖 COMPARAÇÃO DE ALGORITMOS DE CLASSIFICAÇÃO:\n')

# Dados para classificação (sem PnL para evitar data leakage)
X_clf = X_pred  # Features preditivas definidas anteriormente
y_clf = y_true[:len(X_clf)]  # Ajustar tamanho se necessário

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

# Configurar cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Dicionário para armazenar modelos e resultados
models = {}
results = {}

# 1. Random Forest (baseline)
print('1️⃣  Random Forest:')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_scores = cross_val_score(rf, X_train, y_train, cv=cv)
rf_test_score = rf.score(X_test, y_test)
models['Random Forest'] = rf
results['Random Forest'] = {
    'cv_mean': rf_scores.mean(),
    'cv_std': rf_scores.std(),
    'test_score': rf_test_score
}
print(f'   CV Score: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}')
print(f'   Test Score: {rf_test_score:.3f}')

# 2. Gradient Boosting
print('\n2️⃣  Gradient Boosting:')
gb = GradientBoostingClassifier(n_estimators=100, max_depth=6, random_state=42)
gb.fit(X_train, y_train)
gb_scores = cross_val_score(gb, X_train, y_train, cv=cv)
gb_test_score = gb.score(X_test, y_test)
models['Gradient Boosting'] = gb
results['Gradient Boosting'] = {
    'cv_mean': gb_scores.mean(),
    'cv_std': gb_scores.std(),
    'test_score': gb_test_score
}
print(f'   CV Score: {gb_scores.mean():.3f} ± {gb_scores.std():.3f}')
print(f'   Test Score: {gb_test_score:.3f}')

# 3. Neural Network
print('\n3️⃣  Neural Network (MLP):')
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
mlp_scores = cross_val_score(mlp, X_train, y_train, cv=cv)
mlp_test_score = mlp.score(X_test, y_test)
models['Neural Network'] = mlp
results['Neural Network'] = {
    'cv_mean': mlp_scores.mean(),
    'cv_std': mlp_scores.std(),
    'test_score': mlp_test_score
}
print(f'   CV Score: {mlp_scores.mean():.3f} ± {mlp_scores.std():.3f}')
print(f'   Test Score: {mlp_test_score:.3f}')

# 4. Support Vector Machine
print('\n4️⃣  Support Vector Machine:')
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train, y_train)
svm_scores = cross_val_score(svm, X_train, y_train, cv=cv)
svm_test_score = svm.score(X_test, y_test)
models['SVM'] = svm
results['SVM'] = {
    'cv_mean': svm_scores.mean(),
    'cv_std': svm_scores.std(),
    'test_score': svm_test_score
}
print(f'   CV Score: {svm_scores.mean():.3f} ± {svm_scores.std():.3f}')
print(f'   Test Score: {svm_test_score:.3f}')

# 5. Logistic Regression
print('\n5️⃣  Logistic Regression:')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_scores = cross_val_score(lr, X_train, y_train, cv=cv)
lr_test_score = lr.score(X_test, y_test)
models['Logistic Regression'] = lr
results['Logistic Regression'] = {
    'cv_mean': lr_scores.mean(),
    'cv_std': lr_scores.std(),
    'test_score': lr_test_score
}
print(f'   CV Score: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}')
print(f'   Test Score: {lr_test_score:.3f}')

# 6. XGBoost (se disponível)
if HAS_XGB:
    print('\n6️⃣  XGBoost:')
    xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42)
    xgb_clf.fit(X_train, y_train)
    xgb_scores = cross_val_score(xgb_clf, X_train, y_train, cv=cv)
    xgb_test_score = xgb_clf.score(X_test, y_test)
    models['XGBoost'] = xgb_clf
    results['XGBoost'] = {
        'cv_mean': xgb_scores.mean(),
        'cv_std': xgb_scores.std(),
        'test_score': xgb_test_score
    }
    print(f'   CV Score: {xgb_scores.mean():.3f} ± {xgb_scores.std():.3f}')
    print(f'   Test Score: {xgb_test_score:.3f}')

In [ ]:
# 7. Ensemble - Voting Classifier
print('\n7️⃣  Ensemble (Voting Classifier):')
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=6, random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42))
    ],
    voting='soft'
)
voting_clf.fit(X_train, y_train)
voting_scores = cross_val_score(voting_clf, X_train, y_train, cv=cv)
voting_test_score = voting_clf.score(X_test, y_test)
models['Voting Ensemble'] = voting_clf
results['Voting Ensemble'] = {
    'cv_mean': voting_scores.mean(),
    'cv_std': voting_scores.std(),
    'test_score': voting_test_score
}
print(f'   CV Score: {voting_scores.mean():.3f} ± {voting_scores.std():.3f}')
print(f'   Test Score: {voting_test_score:.3f}')

# Ranking dos modelos
print('\n🏆 RANKING POR PERFORMANCE NO TESTE:')
model_ranking = sorted(results.items(), key=lambda x: x[1]['test_score'], reverse=True)
for i, (name, result) in enumerate(model_ranking):
    print(f'   {i+1}. {name:<20} Test: {result["test_score"]:.3f} | CV: {result["cv_mean"]:.3f} ± {result["cv_std"]:.3f}')

In [ ]:
# Visualizar comparação de performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Scores de teste
ax = axes[0]
names = [name for name, _ in model_ranking]
test_scores = [result['test_score'] for _, result in model_ranking]
bars = ax.bar(range(len(names)), test_scores, color=colors[:len(names)])
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Test Accuracy')
ax.set_title('Performance dos Modelos (Teste)', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.0)

# Adicionar valores nas barras
for bar, score in zip(bars, test_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: Cross-validation com error bars
ax = axes[1]
cv_means = [results[name]['cv_mean'] for name in names]
cv_stds = [results[name]['cv_std'] for name in names]
bars = ax.bar(range(len(names)), cv_means, yerr=cv_stds, 
              capsize=5, color=colors[:len(names)], alpha=0.7)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('CV Accuracy')
ax.set_title('Cross-Validation Performance', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

### 8.4 Feature Importance Comparison

In [ ]:
# Comparar feature importance entre modelos que suportam
print('🎯 COMPARAÇÃO DE FEATURE IMPORTANCE:\n')

feature_names = X_clf.columns.tolist()
importance_comparison = pd.DataFrame(index=feature_names)

# Random Forest
rf_importance = models['Random Forest'].feature_importances_
importance_comparison['Random Forest'] = rf_importance

# Gradient Boosting
gb_importance = models['Gradient Boosting'].feature_importances_
importance_comparison['Gradient Boosting'] = gb_importance

# XGBoost (se disponível)
if HAS_XGB and 'XGBoost' in models:
    xgb_importance = models['XGBoost'].feature_importances_
    importance_comparison['XGBoost'] = xgb_importance

# Calcular média e ranking
importance_comparison['Mean'] = importance_comparison.mean(axis=1)
importance_comparison = importance_comparison.sort_values('Mean', ascending=False)

print('Top 10 features por importância média:')
for i, (feature, row) in enumerate(importance_comparison.head(10).iterrows()):
    print(f'  {i+1:2d}. {feature:<30} {row["Mean"]:.3f}')

# Visualizar heatmap de importância
fig, ax = plt.subplots(figsize=(12, 8))
top_features = importance_comparison.head(15)
sns.heatmap(top_features.drop('Mean', axis=1).T, 
            annot=True, fmt='.3f', cmap='YlOrRd', 
            ax=ax, cbar_kws={'label': 'Feature Importance'})
ax.set_title('Feature Importance por Algoritmo (Top 15)', fontsize=14, fontweight='bold')
ax.set_xlabel('Features')
ax.set_ylabel('Algoritmos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 8.5 Confusion Matrix do Melhor Modelo

In [ ]:
# Análise detalhada do melhor modelo
best_model_name = model_ranking[0][0]
best_model = models[best_model_name]
best_score = model_ranking[0][1]['test_score']

print(f'🎯 ANÁLISE DETALHADA: {best_model_name} (Score: {best_score:.3f})\n')

# Predictions
y_pred_best = best_model.predict(X_test)

# Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Accuracy por cluster
print('\nAccuracy por cluster:')
for cluster in sorted(set(y_test)):
    mask = y_test == cluster
    if mask.sum() > 0:
        cluster_accuracy = (y_pred_best[mask] == cluster).mean()
        print(f'  Cluster {cluster}: {cluster_accuracy:.3f} ({mask.sum()} samples)')

### 8.6 Resumo da Comparação

In [ ]:
# Resumo final da comparação
print('🏁 RESUMO DA COMPARAÇÃO DE ALGORITMOS\n')
print('='*60)

# Melhor clustering
print(f'🥇 MELHOR CLUSTERING: {best_name}')
print(f'   Silhouette Score: {best_result["silhouette"]:.3f}')
print(f'   Número de clusters: {best_result["n_clusters"]}')

# Melhor classificador
print(f'\n🥇 MELHOR CLASSIFICADOR: {best_model_name}')
print(f'   Test Accuracy: {best_score:.3f}')
print(f'   CV Score: {results[best_model_name]["cv_mean"]:.3f} ± {results[best_model_name]["cv_std"]:.3f}')

# Top 3 features mais importantes
top3_features = importance_comparison.head(3)
print(f'\n🎯 TOP 3 FEATURES MAIS IMPORTANTES:')
for i, (feature, row) in enumerate(top3_features.iterrows()):
    print(f'   {i+1}. {feature} (importância média: {row["Mean"]:.3f})')

# Insights
print(f'\n💡 INSIGHTS:')
print(f'   • {len(clustering_results)} algoritmos de clustering testados')
print(f'   • {len(models)} algoritmos de classificação comparados')
print(f'   • Melhor performance: {best_score:.1%} de acurácia')
print(f'   • Feature mais discriminante: {top3_features.index[0]}')
clustering_spread = max([r["silhouette"] for r in clustering_results.values()]) - min([r["silhouette"] for r in clustering_results.values()])
classification_spread = max([r["test_score"] for r in results.values()]) - min([r["test_score"] for r in results.values()])
print(f'   • Spread clustering: {clustering_spread:.3f} (diferença entre melhor/pior)')
print(f'   • Spread classificação: {classification_spread:.3f} (diferença entre melhor/pior)')

if clustering_spread < 0.1:
    print('   • Clustering: Algoritmos com performance similar → dados bem estruturados')
else:
    print('   • Clustering: Performance varia significativamente → escolha do algoritmo é crítica')

if classification_spread < 0.05:
    print('   • Classificação: Modelos convergem → padrões consistentes')
else:
    print('   • Classificação: Performance varia → ensemble pode ser vantajoso')

print('='*60)

## 9. Exportar Resultados

In [ ]:
# Salvar resultados principais
results = {
    'trader_features': df_features.to_dict('records'),
    'top200_with_clusters': top200.to_dict('records'),
    'cluster_profiles': cluster_profiles,
    'feature_importance': feature_importance.to_dict('records'),
    'model_accuracy': accuracy,
    'cohort_distribution': cohort_counts,
    'analysis_date': datetime.now().isoformat(),
}

with open('trader_cohorts_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('💾 Resultados salvos em trader_cohorts_results.json')

# CSV para análise externa
df_features.to_csv('trader_features_analysis.csv', index=False)
top200.to_csv('top200_traders_with_clusters.csv', index=False)

print('📊 CSVs exportados para análise externa')
print('   • trader_features_analysis.csv')
print('   • top200_traders_with_clusters.csv')

print('\n🎉 ANÁLISE DE SAFRAS CONCLUÍDA! 🎉')

In [ ]:
# Exportar resultados da comparação de algoritmos
if 'clustering_results' in locals() and 'results' in locals() and isinstance(results, dict):
    algorithm_comparison = {
        'clustering_algorithms': {
            name: {
                'silhouette_score': result.get('silhouette', 0),
                'n_clusters': result.get('n_clusters', 0),
                'algorithm_type': 'clustering'
            } for name, result in clustering_results.items()
        },
        'classification_algorithms': {
            name: {
                'test_score': result.get('test_score', 0),
                'cv_mean': result.get('cv_mean', 0),
                'cv_std': result.get('cv_std', 0),
                'algorithm_type': 'classification'
            } for name, result in results.items() 
            if isinstance(result, dict) and 'test_score' in result
        },
        'best_performers': {
            'clustering': best_name if 'best_name' in locals() else 'K-Means',
            'classification': best_model_name if 'best_model_name' in locals() else 'Random Forest'
        },
        'feature_importance': importance_comparison.head(10).to_dict() if 'importance_comparison' in locals() else {},
        'analysis_timestamp': datetime.now().isoformat()
    }
    
    with open('algorithm_comparison_results.json', 'w') as f:
        json.dump(algorithm_comparison, f, indent=2, default=str)
    
    print('\n🔬 Resultados da comparação de algoritmos salvos em:')
    print('   • algorithm_comparison_results.json')
    
    if 'importance_comparison' in locals():
        importance_comparison.to_csv('feature_importance_comparison.csv')
        print('   • feature_importance_comparison.csv')
else:
    print('\n⚠️  Execute todas as seções anteriores para salvar comparação completa')